# Cipher Swap String Training Pipeline

A minimal 3-step pipeline to train an LLM to swap characters in a strings according to a cipher:

1. **Create env** - Define `CipherSwapEnv` with a binary exact-match reward
2. **Create synthetic data** - Generate 100 random strings with their swaps
3. **Run training job** - Upload and launch the training experiment

## Setup

Install dependencies and configure API credentials.

In [18]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# Configuration
# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = "sk_TOBRulcwDptMqwEBxSsGzHrySPlMBIndvaBXuSjvwXcsNmUjVrCIYIbTNGrjTgKv"
BASE_URL = "https://app.cgft.io"

## Step 1: Create Environment

Define a `CipherSwapEnv` that rewards the model for properly swapping characters. No tools needed — purely a cognitive task.

Definition:
1. "1" swapped with "one"
2. "E" and "e" swapped with "3"
3. "O" amd "o" swapped with "0"
4. "U" swapped with "V"
5. "u" swapped with "v"
6. "S" and "s" swapped with "$"
7. "A" and "a" swapped with "4"
8. "5" swapped with "five"

In [20]:
import re
from pathlib import Path
from typing import Any

from benchmax.envs.base_env import BaseEnv
from benchmax.envs.types import StandardizedExample, ToolDefinition


SYSTEM_PROMPT = """You are a text transformation engine.
Task: Convert all input text using the following cipher. Apply replacements simultaneously to avoid recursive errors (e.g., do not turn "s" into "5" and then into "five").

Replacement Map:
    Numbers: 1 → one, 5 → five
    Vowels: E/e → 3, O/o → 0, U → V, u → v, A/a → 4
    Consonants: S/s → $

Constraint: Write only the output string within the xml tags <answer></answer>.
For example, if the input is "Peace1", your answer should be <answer>P34c3one</answer>."""

async def exact_match_reward(completion: str, ground_truth: str, **kwargs: Any) -> float:
    """Return 1.0 if the <answer> tag content exactly matches the substituted string, else 0.0."""
    if isinstance(completion, list):
        parts = [
            msg.get("content", "")
            for msg in completion
            if isinstance(msg, dict) and msg.get("role") == "assistant"
        ]
        completion = " ".join(parts)

    match = re.search(r"<answer>(.*?)</answer>", completion, re.DOTALL)
    if not match:
        return 0.0
    return 1.0 if match.group(1).strip() == ground_truth else 0.0


class CipherSwapEnv(BaseEnv):
    """Environment for testing LLM string replacement — no tools, binary reward."""

    system_prompt: str = SYSTEM_PROMPT
    reward_funcs = [exact_match_reward]

    def __init__(self, dataset_path: str | None = None, **kwargs):
        self._dataset_path = dataset_path

    def get_train_val_split(self, train_ratio: float = 0.7, seed: int = 42, **kwargs):
        from datasets import load_dataset as hf_load_dataset

        ds = hf_load_dataset(
            "json", data_files=str(Path(self._dataset_path).expanduser().absolute())
        )
        dataset = ds["train"].shuffle(seed=seed)
        split_idx = max(1, min(int(len(dataset) * train_ratio), len(dataset) - 1))
        return dataset.select(range(split_idx)), dataset.select(range(split_idx, len(dataset)))

    @classmethod
    def dataset_preprocess(cls, example: Any, **kwargs) -> StandardizedExample:
        return StandardizedExample(
            prompt=example.get("question", ""),
            ground_truth=example.get("answer", ""),
            init_rollout_args={},
        )

    async def list_tools(self) -> list[ToolDefinition]:
        return []

    async def run_tool(self, rollout_id: str, tool_name: str, **tool_args) -> Any:
        return "Error: SwapCharEnv has no tools"

    async def compute_reward(
        self, rollout_id: str, completion: str, ground_truth: Any, **kwargs: Any
    ) -> dict[str, float]:
        return {"exact_match_reward": await exact_match_reward(completion, ground_truth, **kwargs)}

## Step 2: Create Synthetic Data

Generate synthetic prompt-output pairs. To ensure the model is trained on a variety of inputs, we generate half based on human strings and half based on random assortment of characters.

In [21]:
import random
import string

random.seed(42)

def cipher_transform(text):
    """
    Transforms text by checking each character against a mapping dictionary.
    """
    # Define the mapping for all rules
    cipher_map = {
        "1": "one",
        "5": "five",
        "E": "3", "e": "3",
        "O": "0", "o": "0",
        "U": "V", "u": "v",
        "S": "$", "s": "$",
        "A": "4", "a": "4"
    }
    
    # Process each character: if in map, swap it; otherwise, keep it.
    output = [cipher_map.get(char, char) for char in text]
    
    return "".join(output)

def _rand_str(length: int) -> str:
    chars = string.ascii_letters + string.digits
    return "".join(random.choices(chars, k=length))

# A list of common English words to sample from
word_pool = [
    "status", "success", "active", "attempt", "apples", "oranges", 
    "quick", "brown", "fox", "jumps", "lazy", "dog", "awesome", 
    "everything", "system", "future", "simple", "cipher", "logic", "data"
]

def generate_multi_word_example():
    """Samples 3 words and joins them with spaces."""
    selected_words = random.sample(word_pool, 3)
    return " ".join(selected_words)

def generate_hybrid_dataset(count=100):
    dataset = []
    for _ in range(count):
        # 50% chance of 3-word human strings, 50% random gibberish
        if random.random() > 0.5:
            s = generate_multi_word_example()
        else:
            s = _rand_str(random.randint(5, 15))
            
        dataset.append({
            "question": f"Convert the following string using the cipher rules: {s}",
            "answer": cipher_transform(s) # Uses your reward/ground-truth function
        })
    return dataset

synthetic_dataset = generate_hybrid_dataset(100)

print(f"Generated {len(synthetic_dataset)} examples")
for ex in synthetic_dataset[:3]:
    print(f"  Q: {ex['question']}")
    print(f"  A: {ex['answer']}")


Generated 100 examples
  Q: Convert the following string using the cipher rules: status fox brown
  A: $t4tv$ f0x br0wn
  Q: Convert the following string using the cipher rules: P3fAbn
  A: P3f4bn
  Q: Convert the following string using the cipher rules: status cipher quick
  A: $t4tv$ ciph3r qvick


In [22]:
print(cipher_transform("ilC2eY1gOHaf"))

ilC23Yoneg0H4f


## Step 3: Train Model

Upload the dataset and bundle the environment for training.

In [23]:
from synthetic_data_prep.trainer import train

experiment_id = train(
    env_class=CipherSwapEnv,
    env_args={},
    dataset=synthetic_dataset,
    api_key=API_KEY,
)

print(f"Experiment ID: {experiment_id}")

Auto-generated dataset_id: dataset-444b0776
Uploading dataset (100 items, 11647 bytes)...
Dataset uploaded to: datasets/dataset-444b0776/dataset.jsonl
Bundling CipherSwapEnv...
Pickled class size: 5.24 KB
Metadata size: 0.19 KB
Python version: 3.12
Dependencies: []

Basic local validation CipherSwapEnv in isolated environment (this may take ~1 min)...
[validator] Creating isolated venv...
[validator] Downloading and installing pip...
[validator] Installing dependencies: ['benchmax']
[validator] Dependencies installed successfully.
[validator] Running smoke test...
[validator] OK: CipherSwapEnv instantiated, 0 tools
Isolated validation passed!
Env uploaded to: ~/user-data/envs/08eb09bf/env-cls.pkl

── Remote validation: 2 example(s) on grok-4-fast-reasoning ──

  Example 0 — {"question": "Convert the following string using the cipher rules: status fox brown", "answer": "$t4tv$ f0x br0wn"}
  [ex 0] → rollout_started
  [ex 0] → message [user]: Convert the following string using the cipher

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:
- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse